# 08 · Hands-on capstone — a mini analysis in SQL

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~45 min**

## Learning objectives
- combine everything: SELECT, WHERE, GROUP BY, JOIN, subqueries;
- answer real biological questions with SQL;
- read and interpret your results.

Now you drive. Each step states a question; write the query in the empty cell, then open the solution to check. The steps build a small but complete analysis of the dataset. Work in pairs.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## ⚙️ Setup — run this cell first

This connects the notebook to the example database. It works both on **your own computer** and on **Google Colab** — just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

> ⏱️ ~45 min, in pairs. Try each step before opening its solution.

### Step 1 — Library size
For every sample, compute the total number of reads (`total_reads`), highest first.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, SUM(count) AS total_reads
FROM abundance
GROUP BY sample_id
ORDER BY total_reads DESC;
```
</details>

### Step 2 — Depth per environment
What is the mean library size per environment? (Hint: a CTE, or a subquery in FROM.)

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
WITH sr AS (SELECT sample_id, SUM(count) total_reads FROM abundance GROUP BY sample_id)
SELECT s.environment, ROUND(AVG(sr.total_reads)) AS mean_reads
FROM sr JOIN samples s ON sr.sample_id = s.sample_id
GROUP BY s.environment;
```
</details>

### Step 3 — Community composition
Total reads of each **phylum** in each **environment** (three-table join).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT s.environment, t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
GROUP BY s.environment, t.phylum
ORDER BY s.environment, reads DESC;
```
</details>

### Step 4 — Treatment effect
Compare the total reads of the phylum `Cyanobacteriota` between the two treatments.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT s.treatment, SUM(a.count) AS cyano_reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
WHERE t.phylum = 'Cyanobacteriota'
GROUP BY s.treatment;
```
</details>

### Step 5 — Prevalence
In how many distinct samples does each phylum appear? (Most widespread first.)

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT t.phylum, COUNT(DISTINCT a.sample_id) AS n_samples
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.phylum
ORDER BY n_samples DESC;
```
</details>

### Step 6 — Acidic soils
List the acidic samples (pH < 6.5) together with the phylum that has the most reads in each of them. (Hint: join, group by sample and phylum, and use a window rank or per-sample ordering.)

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
WITH ranked AS (
  SELECT s.sample_id, s.ph, t.phylum, SUM(a.count) AS reads,
         RANK() OVER (PARTITION BY s.sample_id ORDER BY SUM(a.count) DESC) AS rnk
  FROM abundance a
  JOIN samples s ON a.sample_id = s.sample_id
  JOIN taxa t    ON a.taxon_id  = t.taxon_id
  WHERE s.ph < 6.5
  GROUP BY s.sample_id, t.phylum
)
SELECT sample_id, ph, phylum AS dominant_phylum, reads
FROM ranked WHERE rnk = 1
ORDER BY ph;
```
</details>

### Step 7 — Composition as percentages
Turn Step 3 into **relative abundance**: the percentage each phylum contributes *within* each environment (chained CTEs, or a window aggregate).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
WITH ep AS (
  SELECT s.environment, t.phylum, SUM(a.count) AS reads
  FROM abundance a JOIN samples s ON a.sample_id = s.sample_id
  JOIN taxa t ON a.taxon_id = t.taxon_id
  GROUP BY s.environment, t.phylum),
et AS (SELECT environment, SUM(reads) total FROM ep GROUP BY environment)
SELECT ep.environment, ep.phylum, ROUND(100.0*ep.reads/et.total,1) AS pct
FROM ep JOIN et ON ep.environment = et.environment
ORDER BY ep.environment, pct DESC;
```
</details>

### ✍️ Interpretation (double-click to write)
- Which environment is richest in reads, and which phyla define each environment?
- Did the Amended treatment change the Cyanobacteria?
- One question you would ask this database next: …

### Recap
- You built a full analysis with `SELECT`, `WHERE`, `GROUP BY`, `JOIN`, CTEs and windows.
- The same queries work on a real study — just point them at a real database.
- Next: **09 · Bonus — creating & changing data**.